Задание 1 (синтаксис)

Для каждого покупателя получите таблицу со столбцами:
 • preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
 • total_spent - сумма всех трат;
 • addons_spent - сумма на дополнительные услуги и аксессуары;
 • orders_count - число заказов.
 
 Отсортируйте по total_spent по убыванию и выведите топ-10.

Пример вывода (формат; числа зависят от датасета):

   customer_id  preferred_payment  total_spent  addons_spent  orders_count
 0         C042     Credit Card              15420.50        890.00             7
 1         C018     PayPal                     14810.00        450.00             5
 2         C103     Debit Card               13990.25       1200.00             6
 ...

Пограничные случаи:

• Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
 • Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
 • Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
 • Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.

In [78]:
import pandas as pd

e_sales_path = "./Electronic_sales_Sep2023-Sep2024.csv"

e_sales = pd.read_csv(e_sales_path)

#  Фильтр пустых значений в Total Price и Payment Method
filtered_e_sales = e_sales.loc[
    e_sales["Total Price"].notna()
    & e_sales["Payment Method"].notna()
    & e_sales["Order Status"].eq(
        "Completed"
    )  #  Вероятно имелось в виду успешных заказов
].copy()
filtered_e_sales["Payment Method"] = filtered_e_sales["Payment Method"].replace(
    {"Paypal": "PayPal"}
)  #  При проверке через AI говорит PayPal тут два разных :)

preferred_methods = (
    filtered_e_sales.groupby(["Customer ID", "Payment Method"])
    .agg(method_order_count=("Customer ID", "count"))
    .sort_values(
        ["Customer ID", "method_order_count", "Payment Method"],
        ascending=[True, False, True],
    )  #  Берется первый метод по алфавиту
    .reset_index()
    .drop_duplicates("Customer ID")
    .drop("method_order_count", axis=1)
)

mid_customer_stats = filtered_e_sales.groupby("Customer ID").agg(
    total_spent=("Total Price", "sum"),
    addons_spent=("Add-on Total", "sum"),
    orders_count=("Customer ID", "count"),
)

customer_stats = mid_customer_stats.merge(
    preferred_methods, on="Customer ID", how="left"
)

top_sorted_customer_stats = (
    customer_stats.sort_values("total_spent", ascending=False).rename(
        columns={"Customer ID": "customer_id", "Payment Method": "preferred_payment"}
    )
).head(10)

print(top_sorted_customer_stats)

      customer_id  total_spent  addons_spent  orders_count preferred_payment
5048        11476     29937.93        198.42             4            PayPal
7095        15399     29084.88        305.39             3            PayPal
6177        13635     28093.28        388.11             4       Credit Card
7584        16357     27486.01        352.75             6     Bank Transfer
4960        11332     26725.65        240.24             4            PayPal
5489        12294     26575.02        477.68             4     Bank Transfer
839          2556     26098.40        230.66             6        Debit Card
6124        13534     25893.60        367.61             5     Bank Transfer
6589        14436     24912.42        102.31             4       Credit Card
6266        13797     24870.42         78.83             3     Bank Transfer
